## 1. Import libraries and set reproducibility

In [4]:
import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from tensorflow import keras
from tensorflow.keras import layers, regularizers
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.utils import plot_model

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow version:", tf.__version__)
print("GPU devices:", tf.config.list_physical_devices("GPU"))

TensorFlow version: 2.21.0
GPU devices: []


## 2. Load CIFAR-100 and prepare the data

In [7]:

# CIFAR-100 contains 50,000 training images and 10,000 test images.
# Each image is 32x32 pixels with 3 colour channels and belongs to one of 100 classes.
(x_train, y_train), (x_test, y_test) = keras.datasets.cifar100.load_data(label_mode="fine")

print("Training images:", x_train.shape)
print("Training labels:", y_train.shape)
print("Test images:", x_test.shape)
print("Test labels:", y_test.shape)
print("Number of classes:", len(np.unique(y_train)))

169001437/169001437 ━━━━━━━━━━━━━━━━━━━━ 1604s 9us/step
Training images: (50000, 32, 32, 3)
Training labels: (50000, 1)
Test images: (10000, 32, 32, 3)
Test labels: (10000, 1)
Number of classes: 100


#**Image normalizing**


In [8]:

# Normalize training images by dividing pixel values by 255
x_train = x_train.astype('float32') / 255.0

# Normalize testing images by dividing pixel values by 255
x_test = x_test.astype('float32') / 255.0

# Display the shape of training images
print("Training data shape:", x_train.shape)

# Display the shape of testing images
print("Testing data shape:", x_test.shape)

# Display minimum and maximum pixel values after normalization
print("Minimum pixel value:", x_train.min())
print("Maximum pixel value:", x_train.max())

Training data shape: (50000, 32, 32, 3)
Testing data shape: (10000, 32, 32, 3)
Minimum pixel value: 0.0
Maximum pixel value: 1.0


#**Build CNN model**

In [9]:
# Import TensorFlow and Keras layers
import tensorflow as tf
from tensorflow.keras import layers, models

# Create a Sequential CNN model
model = models.Sequential([

    # Define input image size: 32x32 RGB
    layers.Input(shape=(32, 32, 3)),

    # First convolutional layer
    layers.Conv2D(32, (3, 3), activation='relu', padding='same'),

    # Normalize layer outputs for stable training
    layers.BatchNormalization(),

    # Reduce image dimensions
    layers.MaxPooling2D((2, 2)),

    # Second convolutional layer
    layers.Conv2D(64, (3, 3), activation='relu', padding='same'),

    # Normalize layer outputs
    layers.BatchNormalization(),

    # Reduce image dimensions
    layers.MaxPooling2D((2, 2)),

    # Third convolutional layer
    layers.Conv2D(128, (3, 3), activation='relu', padding='same'),

    # Normalize layer outputs
    layers.BatchNormalization(),

    # Reduce image dimensions
    layers.MaxPooling2D((2, 2)),

    # Convert feature maps into a one-dimensional vector
    layers.Flatten(),

    # Fully connected layer
    layers.Dense(256, activation='relu'),

    # Dropout to reduce overfitting
    layers.Dropout(0.5),

    # Output layer for 100 CIFAR-100 classes
    layers.Dense(100, activation='softmax')
])

# Display the CNN architecture and parameter counts
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 32, 32, 32)     │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 32, 32, 32)     │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 16, 16, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 16, 16, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 16, 16, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 8, 8, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 8, 8, 128)      │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 8, 8, 128)      │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 4, 4, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 2048)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │       524,544 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 100)            │        25,700 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 644,388 (2.46 MB)

 Trainable params: 643,940 (2.46 MB)

 Non-trainable params: 448 (1.75 KB)

In [10]:
# Compile the CNN model

model.compile(

    # Adam optimizer updates model weights during training
    optimizer='adam',

    # Loss function for 100-class classification
    loss='sparse_categorical_crossentropy',

    # Measure model accuracy during training
    metrics=['accuracy']
)

# Display confirmation message
print("CNN model compiled successfully!")

CNN model compiled successfully!


In [ ]:
# Train the CNN model using the training dataset

history = model.fit(x_train,y_train,epochs=10,batch_size=128,validation_split=0.2,verbose=1)

# Print a message after training finishes
print("CNN model training completed!")

Epoch 1/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 165s 515ms/step - accuracy: 0.0608 - loss: 4.2958 - val_accuracy: 0.0126 - val_loss: 12.0864
Epoch 2/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 153s 490ms/step - accuracy: 0.1246 - loss: 3.7462 - val_accuracy: 0.1540 - val_loss: 3.5563
Epoch 3/10
279/313 ━━━━━━━━━━━━━━━━━━━━ 16s 480ms/step - accuracy: 0.1578 - loss: 3.5002

In [ ]:
# Import matplotlib for plotting graphs
import matplotlib.pyplot as plt

# Create a figure with two graphs
plt.figure(figsize=(12, 5))

# Plot training and validation accuracy
plt.subplot(1, 2, 1)

# Training accuracy for each epoch
plt.plot(history.history['accuracy'], label='Training Accuracy')

# Validation accuracy for each epoch
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')

# Add graph title and labels
plt.title('CNN Model Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')

# Display legend
plt.legend()

# Plot training and validation loss
plt.subplot(1, 2, 2)

# Training loss for each epoch
plt.plot(history.history['loss'], label='Training Loss')

# Validation loss for each epoch
plt.plot(history.history['val_loss'], label='Validation Loss')

# Add graph title and labels
plt.title('CNN Model Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')

# Display legend
plt.legend()

# Adjust spacing between graphs
plt.tight_layout()

# Display both graphs
plt.show()

In [ ]:
# Evaluate the trained CNN model using test images

test_loss, test_accuracy = model.evaluate(
    x_test,
    y_test,
    verbose=1
)

# Print the final test loss
print("Test Loss:", test_loss)

# Print the final test accuracy as a percentage
print("Test Accuracy: {:.2f}%".format(test_accuracy * 100))

In [ ]:
# Import required evaluation metrics
from sklearn.metrics import classification_report

# Predict the classes of test images
y_pred_prob = model.predict(x_test)

# Convert predicted probabilities into class labels
y_pred = y_pred_prob.argmax(axis=1)

# Convert actual labels into a one-dimensional array
y_true = y_test.flatten()

# Display precision, recall, and F1-score for all 100 classes
print(classification_report(
    y_true,
    y_pred,
    digits=4,
    zero_division=0
))

In [ ]:
#callbacks
# Import the required Keras callbacks
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

# Stop training if validation loss does not improve for 3 epochs
early_stop = EarlyStopping( monitor='val_loss',
    patience=3,
    restore_best_weights=True,
    verbose=1
)

# Save the model whenever validation loss improves
checkpoint = ModelCheckpoint('best_cnn_model.keras',
    monitor='val_loss',
    save_best_only=True,
    verbose=1
)

# Train the CNN model using both callbacks
history_callback = model.fit(x_train,
    y_train,
    epochs=10,
    batch_size=128,
    validation_split=0.2,
    callbacks=[early_stop, checkpoint],
    verbose=1
)

# Display a confirmation message
print("Training with callbacks completed!")

#**CNN architecture**

In [ ]:
# Import the model plotting function
from tensorflow.keras.utils import plot_model

# Generate the CNN architecture diagram
plot_model(
    cnn_model,
    to_file="cnn_architecture.png",
    show_shapes=True,
    show_layer_names=True,
    show_layer_activations=True,
    dpi=100
)

# Display the architecture image
from IPython.display import Image, display
display(Image(filename="cnn_architecture.png"))

#**VGG-16 model**

In [ ]:
# Import required libraries
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

from tensorflow.keras import layers, models

# Prepare labels as integer values
y_train = y_train.reshape(-1).astype("int32")
y_test = y_test.reshape(-1).astype("int32")

# Apply VGG-16 preprocessing
# The original images are already normalized to 0-1
x_train_vgg = tf.keras.applications.vgg16.preprocess_input(
    x_train * 255.0
)

x_test_vgg = tf.keras.applications.vgg16.preprocess_input(
    x_test * 255.0
)

# Display dataset shapes
print("Training images:", x_train_vgg.shape)
print("Testing images:", x_test_vgg.shape)
print("Training labels:", y_train.shape)
print("Testing labels:", y_test.shape)

#**Create VGG model**

In [ ]:
# Load the pretrained VGG-16 convolutional base
vgg_base = tf.keras.applications.VGG16(
    weights="imagenet",
    include_top=False,
    input_shape=(32, 32, 3)
)

# Freeze the pretrained convolutional layers
vgg_base.trainable = False

# Create the VGG-16 model for CIFAR-100 classification
vgg16_model = models.Sequential([
    layers.Input(shape=(32, 32, 3)),
    vgg_base,
    layers.GlobalAveragePooling2D(),
    layers.Dense(256, activation="relu"),
    layers.Dropout(0.4),
    layers.Dense(100, activation="softmax")
])

# Compile the model
vgg16_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

# Display the model architecture
vgg16_model.summary()


In [ ]:
#training
# Create an EarlyStopping callback
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True
)

# Train the VGG-16 model
vgg_history = vgg16_model.fit(
    x_train_vgg,
    y_train,
    epochs=10,
    batch_size=64,
    validation_split=0.2,
    callbacks=[early_stopping],
    verbose=1
)

In [ ]:
# Evaluate the trained VGG-16 model on test data
vgg_test_loss, vgg_test_accuracy = vgg16_model.evaluate(
    x_test_vgg,
    y_test,
    verbose=1
)

# Display the evaluation results
print("VGG-16 Test Loss:", round(vgg_test_loss, 4))
print("VGG-16 Test Accuracy:", round(vgg_test_accuracy * 100, 2), "%")

In [ ]:
# Import evaluation metrics
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix
import numpy as np

# Generate predictions on the test dataset
y_pred_prob = vgg16_model.predict(x_test_vgg)

# Convert predicted probabilities to class labels
y_pred = np.argmax(y_pred_prob, axis=1)

# Display the classification report
print("VGG-16 Classification Report:")
print(classification_report(
    y_test,
    y_pred,
    digits=4,
    zero_division=0
))

In [ ]:
# Plot VGG-16 training and validation accuracy
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(vgg_history.history["accuracy"], label="Training Accuracy")
plt.plot(vgg_history.history["val_accuracy"], label="Validation Accuracy")
plt.title("VGG-16 Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()

# Plot VGG-16 training and validation loss
plt.subplot(1, 2, 2)
plt.plot(vgg_history.history["loss"], label="Training Loss")
plt.plot(vgg_history.history["val_loss"], label="Validation Loss")
plt.title("VGG-16 Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()

plt.tight_layout()
plt.show()

#**VGG architecture**

In [ ]:
# Generate the VGG-16 architecture diagram
plot_model(
    vgg16_model,
    to_file="vgg16_architecture.png",
    show_shapes=True,
    show_layer_names=True,
    show_layer_activations=True,
    dpi=100
)

# Display the architecture image
display(Image(filename="vgg16_architecture.png"))

#**CNN VGG comparison graph**

In [ ]:
# Compare the test accuracy of both models
import matplotlib.pyplot as plt

model_names = ["Custom CNN", "VGG-16"]
test_accuracies = [37.15, 33.38]

# Plot the accuracy comparison
plt.figure(figsize=(8, 5))
bars = plt.bar(model_names, test_accuracies)

plt.title("CIFAR-100: CNN vs VGG-16")
plt.xlabel("Model")
plt.ylabel("Test Accuracy (%)")
plt.ylim(0, 50)

# Display accuracy values above the bars
for bar, accuracy in zip(bars, test_accuracies):
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.5,
        f"{accuracy:.2f}%",
        ha="center"
    )

plt.tight_layout()
plt.show()

##**Conclusion**
The Custom CNN achieved 3.77 percentage points higher test accuracy than VGG-16. It also obtained a lower test loss and higher macro precision, recall, and F1-score. Therefore, the Custom CNN performed better than the VGG-16 transfer-learning model under the experimental settings used.

##**Overall Conclusion**
This project compared a custom CNN and a pretrained VGG-16 model for image classification on the CIFAR-100 dataset. The Custom CNN achieved a test accuracy of 37.15%, while VGG-16 achieved 33.38%. Based on these results, the Custom CNN was the better-performing model in this experiment. However, both models have scope for improvement because CIFAR-100 contains 100 classes with visually diverse images. Future improvements could include data augmentation, learning-rate scheduling, hyperparameter tuning, and fine-tuning the pretrained VGG-16 layers. These experiments demonstrate the application of convolutional neural networks and transfer learning to multi-class image classification.

